# NB09 — Can LLMs Be Reliable Coaches?

## Measuring Inter-Model Agreement, Scoring Consistency, and Feedback Quality When Evaluating Open-Ended Product Analytics Responses

**Data Story Premise:** We used Claude, GPT-4o-mini, and Gemini as automated coaches for product analytics interview prep. Each model independently scored the same open-ended responses across 7 framework steps. This notebook analyzes **how well these models agree with each other**, where they diverge, and what that tells us about using LLMs as evaluation tools for unstructured analytical reasoning.

**Key questions:**
1. **Inter-rater reliability:** Do the three models agree on what a "good" answer looks like? (Krippendorff's alpha, ICC)
2. **Scoring distributions:** Are some models consistently harsher or more lenient? (systematic bias)
3. **Dimension-level agreement:** Do models agree more on some rubric dimensions than others?
4. **Step difficulty:** Which framework steps produce the most model disagreement?
5. **Score-length correlation:** Do longer responses get higher scores? (a known LLM bias)
6. **Latency vs. quality:** Does response time correlate with scoring quality?
7. **Feedback quality:** How specific and actionable is the qualitative feedback?

---

In [ ]:
import os, sys, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import interview_practice_utils as ipu

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
NB08_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'nb08')
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'nb09')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f"Loading session data from: {NB08_DIR}")
df = ipu.load_all_sessions(NB08_DIR)

if df.empty:
    display(HTML('''
    <div style="background:#fef2f2; border-left:4px solid #dc2626; padding:16px; border-radius:6px;">
        <h3 style="margin:0 0 8px 0;">No Practice Session Data Found</h3>
        <p>Run <strong>NB08</strong> first to generate practice sessions. The more sessions you complete,
        the richer this analysis will be.</p>
        <p><strong>Recommended:</strong> Complete at least 3-5 full sessions (all 7 steps each) with
        multiple LLM models enabled for meaningful inter-rater analysis.</p>
    </div>'''))
else:
    n_sessions = df['session_id'].nunique()
    n_steps = len(df)
    models_present = [c.replace('_composite', '') for c in df.columns if c.endswith('_composite') and df[c].notna().any()]
    print(f"Sessions: {n_sessions} | Step records: {n_steps} | Models with scores: {models_present}")
    display(df.head())


## 1. Data Preparation

Reshape the session log into analysis-ready formats: long-form (one row per model-step) for agreement analysis, and wide-form for correlation analysis.

In [ ]:
# Identify which models have composite scores
MODEL_COLS = {}
for col in df.columns:
    if col.endswith('_composite') and col != 'composite_score':
        model = col.replace('_composite', '')
        if df[col].notna().sum() > 0:
            MODEL_COLS[model] = col

print(f"Models detected: {list(MODEL_COLS.keys())}")

# Create long-form: one row per (session, step, model)
long_rows = []
for _, row in df.iterrows():
    for model, col in MODEL_COLS.items():
        if pd.notna(row.get(col)):
            long_rows.append({
                'session_id': row['session_id'],
                'scenario_id': row['scenario_id'],
                'archetype': row.get('archetype', ''),
                'situation': row.get('situation', ''),
                'scope': row.get('scope', ''),
                'step_num': row['step_num'],
                'step_name': row['step_name'],
                'model': model,
                'composite_score': row[col],
                'latency': row.get(f'{model}_latency', None),
                'consensus_score': row['composite_score'],
                'response_length': row.get('user_response_length', 0),
            })

df_long = pd.DataFrame(long_rows)
print(f"Long-form dataset: {len(df_long)} rows")
if not df_long.empty:
    display(df_long.head(10))


## 2. Scoring Distributions by Model

**Question:** Are some models systematically harsher or more lenient?

If one model consistently scores 0.5 points higher than others, that's a systematic bias we need to account for. In traditional inter-rater reliability, we'd call this "rater severity."


In [ ]:
if not df_long.empty and len(MODEL_COLS) >= 1:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 2a. Distribution overlay
    ax = axes[0]
    for model in MODEL_COLS:
        subset = df_long[df_long['model'] == model]['composite_score']
        if len(subset) > 0:
            ax.hist(subset, bins=np.arange(0.5, 5.75, 0.5), alpha=0.5, label=f'{model} (μ={subset.mean():.2f})')
    ax.set_xlabel('Composite Score')
    ax.set_ylabel('Count')
    ax.set_title('Score Distributions by Model')
    ax.legend()
    ax.set_xlim(0.5, 5.5)

    # 2b. Box plot
    ax = axes[1]
    models_list = list(MODEL_COLS.keys())
    box_data = [df_long[df_long['model'] == m]['composite_score'].dropna() for m in models_list]
    bp = ax.boxplot(box_data, labels=models_list, patch_artist=True,
                    boxprops=dict(facecolor='#dbeafe'))
    ax.set_ylabel('Composite Score')
    ax.set_title('Score Range by Model')
    ax.set_ylim(0, 5.5)

    # 2c. Mean scores by step and model
    ax = axes[2]
    pivot = df_long.groupby(['step_num', 'model'])['composite_score'].mean().unstack()
    if not pivot.empty:
        pivot.plot(kind='bar', ax=ax, width=0.7)
        ax.set_xlabel('Step Number')
        ax.set_ylabel('Mean Composite Score')
        ax.set_title('Mean Score by Step × Model')
        ax.legend(title='Model', fontsize=9)
        ax.set_ylim(0, 5.5)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, 'model_score_distributions.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Summary stats
    print("\n=== Model Scoring Summary ===")
    summary = df_long.groupby('model')['composite_score'].agg(['mean', 'std', 'min', 'max', 'count'])
    display(summary.round(2))

    # Kruskal-Wallis test for significant differences
    if len(MODEL_COLS) >= 2:
        groups = [df_long[df_long['model'] == m]['composite_score'].dropna().values for m in models_list]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"\nKruskal-Wallis test: H={stat:.2f}, p={p:.4f}")
            if p < 0.05:
                print("→ Significant difference in scoring severity between models.")
            else:
                print("→ No significant difference — models score at similar levels.")
else:
    print("Insufficient data for distribution analysis. Complete more NB08 sessions.")


## 3. Inter-Rater Reliability

**Question:** Do the models agree on which responses are good vs. bad?

We use three measures:

- **Krippendorff's alpha (α):** The gold standard for inter-rater reliability. α > 0.8 = reliable, 0.67-0.8 = acceptable, < 0.67 = questionable.
- **Intraclass Correlation Coefficient (ICC):** Measures consistency of ratings across raters. ICC > 0.75 = good, 0.5-0.75 = moderate, < 0.5 = poor.
- **Pairwise Pearson correlations:** How linearly related are each pair of models' scores?


In [ ]:
def krippendorff_alpha(data_matrix):
    """
    Compute Krippendorff's alpha for interval data.
    data_matrix: numpy array (n_raters × n_items), NaN for missing.
    """
    # Remove items with fewer than 2 raters
    valid_mask = ~np.isnan(data_matrix)
    n_raters_per_item = valid_mask.sum(axis=0)
    keep = n_raters_per_item >= 2
    data = data_matrix[:, keep]
    valid = ~np.isnan(data)
    n_items = data.shape[1]

    if n_items < 2:
        return np.nan

    # Observed disagreement
    Do = 0
    n_pairs = 0
    for j in range(n_items):
        vals = data[valid[:, j], j]
        m = len(vals)
        if m < 2:
            continue
        for a in range(m):
            for b in range(a + 1, m):
                Do += (vals[a] - vals[b]) ** 2
                n_pairs += 1
    if n_pairs == 0:
        return np.nan
    Do /= n_pairs

    # Expected disagreement
    all_vals = data[valid]
    n_total = len(all_vals)
    De = 0
    n_epairs = 0
    # Use sampling for efficiency if large
    if n_total > 500:
        rng = np.random.RandomState(42)
        idx = rng.choice(n_total, min(500, n_total), replace=False)
        sample = all_vals[idx]
    else:
        sample = all_vals
    for a in range(len(sample)):
        for b in range(a + 1, len(sample)):
            De += (sample[a] - sample[b]) ** 2
            n_epairs += 1
    if n_epairs == 0:
        return np.nan
    De /= n_epairs

    if De == 0:
        return 1.0
    return 1 - Do / De


if not df_long.empty and len(MODEL_COLS) >= 2:
    models_list = list(MODEL_COLS.keys())

    # Build rater × item matrix
    # Each "item" = unique (session_id, step_num)
    items = df[['session_id', 'step_num']].drop_duplicates().reset_index(drop=True)
    item_ids = {(r['session_id'], r['step_num']): i for i, r in items.iterrows()}

    data_matrix = np.full((len(models_list), len(items)), np.nan)
    for _, row in df_long.iterrows():
        item_key = (row['session_id'], row['step_num'])
        if item_key in item_ids:
            model_idx = models_list.index(row['model'])
            data_matrix[model_idx, item_ids[item_key]] = row['composite_score']

    alpha = krippendorff_alpha(data_matrix)

    # ICC (two-way random, absolute agreement)
    # Using wide format
    wide = df_long.pivot_table(index=['session_id', 'step_num'], columns='model',
                                values='composite_score').dropna()
    if len(wide) >= 3 and len(wide.columns) >= 2:
        n = len(wide)
        k = len(wide.columns)
        grand_mean = wide.values.mean()
        row_means = wide.values.mean(axis=1)
        col_means = wide.values.mean(axis=0)
        SS_total = np.sum((wide.values - grand_mean) ** 2)
        SS_rows = k * np.sum((row_means - grand_mean) ** 2)
        SS_cols = n * np.sum((col_means - grand_mean) ** 2)
        SS_error = SS_total - SS_rows - SS_cols
        MS_rows = SS_rows / (n - 1)
        MS_cols = SS_cols / (k - 1) if k > 1 else 0
        MS_error = SS_error / ((n - 1) * (k - 1)) if (n - 1) * (k - 1) > 0 else 0
        icc = (MS_rows - MS_error) / (MS_rows + (k - 1) * MS_error + k * (MS_cols - MS_error) / n) if (MS_rows + (k - 1) * MS_error + k * (MS_cols - MS_error) / n) != 0 else 0
    else:
        icc = np.nan

    # Pairwise correlations
    print("=== Inter-Rater Reliability ===\n")
    print(f"Krippendorff's alpha: {alpha:.3f}", end='')
    if alpha >= 0.8:
        print(" (RELIABLE)")
    elif alpha >= 0.667:
        print(" (ACCEPTABLE)")
    else:
        print(" (QUESTIONABLE)")

    print(f"ICC (two-way, absolute): {icc:.3f}", end='')
    if icc >= 0.75:
        print(" (GOOD)")
    elif icc >= 0.5:
        print(" (MODERATE)")
    else:
        print(" (POOR)")

    if len(wide.columns) >= 2:
        print("\nPairwise Pearson Correlations:")
        corr_matrix = wide.corr()
        display(corr_matrix.round(3))

        # Heatmap
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
                    vmin=0, vmax=1, center=0.5, ax=ax,
                    square=True, linewidths=1)
        ax.set_title('Pairwise Model Score Correlations')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUTS_DIR, 'model_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("Need scores from at least 2 models for inter-rater analysis.")
    print("Enable multiple API keys in NB08 and run more sessions.")


## 4. Step Difficulty & Model Disagreement

**Question:** Which framework steps produce the most disagreement between models?

Steps where models disagree strongly may represent dimensions of analytical reasoning that LLMs evaluate differently — interesting for the data story because it maps to where human evaluators might also disagree.


In [ ]:
if not df_long.empty and len(MODEL_COLS) >= 2:
    # Score spread per step
    spread = df.groupby('step_num').agg(
        mean_score=('composite_score', 'mean'),
        agreement_std=('agreement_std_dev', 'mean'),
        agreement_range=('agreement_range', 'mean'),
        n_responses=('composite_score', 'count'),
    ).round(2)

    step_names = {s['step_num']: s['step_name'] for s in ipu.FRAMEWORK_STEPS}
    spread['step_name'] = spread.index.map(step_names)

    print("=== Step Difficulty & Agreement ===\n")
    display(spread[['step_name', 'mean_score', 'agreement_std', 'agreement_range', 'n_responses']])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Mean score by step
    ax = axes[0]
    colors = ['#dc2626' if s < 3.0 else '#f59e0b' if s < 4.0 else '#16a34a'
              for s in spread['mean_score']]
    ax.barh(spread['step_name'], spread['mean_score'], color=colors)
    ax.set_xlabel('Mean Composite Score')
    ax.set_title('Average Score by Framework Step')
    ax.set_xlim(0, 5)
    ax.invert_yaxis()

    # Agreement spread by step
    ax = axes[1]
    ax.barh(spread['step_name'], spread['agreement_std'], color='#6366f1')
    ax.set_xlabel('Mean Inter-Model Std Dev')
    ax.set_title('Model Disagreement by Framework Step\n(higher = more disagreement)')
    ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, 'step_difficulty_agreement.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Need multi-model data for step difficulty analysis.")


## 5. Response Length vs. Score (LLM Bias Detection)

**Question:** Do longer responses receive higher scores regardless of quality?

This is a known LLM evaluation bias. If we find a strong positive correlation between response length and score, it suggests the models may be rewarding verbosity rather than analytical quality.


In [ ]:
if not df_long.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter: length vs consensus score
    ax = axes[0]
    ax.scatter(df['user_response_length'], df['composite_score'],
               alpha=0.5, color='#2563eb', edgecolors='white', s=60)
    # Regression line
    mask = df[['user_response_length', 'composite_score']].dropna()
    if len(mask) >= 3:
        slope, intercept, r, p, se = stats.linregress(mask['user_response_length'], mask['composite_score'])
        x_line = np.linspace(mask['user_response_length'].min(), mask['user_response_length'].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, 'r--', linewidth=2,
                label=f'r={r:.2f}, p={p:.3f}')
        ax.legend()
    ax.set_xlabel('Response Length (characters)')
    ax.set_ylabel('Consensus Composite Score')
    ax.set_title('Does Verbosity = Higher Scores?')
    ax.set_ylim(0, 5.5)

    # Per-model length bias
    ax = axes[1]
    for model in MODEL_COLS:
        sub = df_long[df_long['model'] == model].dropna(subset=['composite_score'])
        if len(sub) >= 3:
            r, p = stats.pearsonr(sub['response_length'], sub['composite_score'])
            ax.bar(model, r, color=['#dc2626' if abs(r) > 0.5 else '#f59e0b' if abs(r) > 0.3 else '#16a34a'])
            ax.text(model, r + 0.02, f'r={r:.2f}', ha='center', fontsize=10)
    ax.set_ylabel('Pearson r (length vs score)')
    ax.set_title('Length Bias by Model\n(closer to 0 = less biased)')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylim(-1, 1)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, 'length_bias_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print("\nInterpretation:")
    if len(mask) >= 3:
        if abs(r) > 0.5:
            print(f"  Strong length-score correlation (r={r:.2f}) — models may be rewarding verbosity.")
        elif abs(r) > 0.3:
            print(f"  Moderate length-score correlation (r={r:.2f}) — some length bias present.")
        else:
            print(f"  Weak length-score correlation (r={r:.2f}) — models are evaluating content over length.")
else:
    print("Need session data for length-bias analysis.")


## 6. Model Latency Comparison

**Question:** How do the models compare on response time, and does latency correlate with score quality?

Relevant for production use: if one model takes 3× longer but doesn't score differently, the faster model is preferable for a coaching tool.


In [ ]:
if not df_long.empty and df_long['latency'].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Latency distribution
    ax = axes[0]
    for model in MODEL_COLS:
        sub = df_long[(df_long['model'] == model) & df_long['latency'].notna()]
        if len(sub) > 0:
            ax.hist(sub['latency'], bins=15, alpha=0.5, label=f'{model} (μ={sub["latency"].mean():.1f}s)')
    ax.set_xlabel('Latency (seconds)')
    ax.set_ylabel('Count')
    ax.set_title('Scoring Latency Distribution by Model')
    ax.legend()

    # Latency box plot
    ax = axes[1]
    lat_data = []
    lat_labels = []
    for model in MODEL_COLS:
        sub = df_long[(df_long['model'] == model) & df_long['latency'].notna()]['latency']
        if len(sub) > 0:
            lat_data.append(sub)
            lat_labels.append(model)
    if lat_data:
        ax.boxplot(lat_data, labels=lat_labels, patch_artist=True,
                   boxprops=dict(facecolor='#fef3c7'))
        ax.set_ylabel('Latency (seconds)')
        ax.set_title('Latency Range by Model')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, 'model_latency.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Summary stats
    print("\n=== Latency Summary (seconds) ===")
    lat_summary = df_long.groupby('model')['latency'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
    display(lat_summary.round(2))
else:
    print("No latency data available.")


## 7. Learning Curve — Score Progression Over Sessions

**Question:** Do scores improve with practice?

If the coaching tool is effective, we should see an upward trend in composite scores across sessions. This is the most compelling metric for the data story's conclusion.


In [ ]:
if not df.empty and df['session_id'].nunique() >= 2:
    # Order sessions by timestamp
    session_order = df.groupby('session_id')['timestamp'].min().sort_values()
    session_map = {sid: i + 1 for i, sid in enumerate(session_order.index)}
    df['session_num'] = df['session_id'].map(session_map)

    session_avg = df.groupby('session_num')['composite_score'].mean()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Overall progression
    ax = axes[0]
    ax.plot(session_avg.index, session_avg.values, 'o-', color='#2563eb',
            linewidth=2, markersize=8)
    if len(session_avg) >= 3:
        z = np.polyfit(session_avg.index, session_avg.values, 1)
        p = np.poly1d(z)
        ax.plot(session_avg.index, p(session_avg.index), 'r--', alpha=0.7,
                label=f'Trend: {z[0]:+.2f}/session')
        ax.legend()
    ax.set_xlabel('Session Number')
    ax.set_ylabel('Mean Composite Score')
    ax.set_title('Learning Curve — Score Progression')
    ax.set_ylim(0, 5.5)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # Per-step progression
    ax = axes[1]
    step_colors = plt.cm.Set2(np.linspace(0, 1, 7))
    for step_num in range(1, 8):
        sub = df[df['step_num'] == step_num].groupby('session_num')['composite_score'].mean()
        if len(sub) >= 2:
            step_name = ipu.FRAMEWORK_STEPS[step_num - 1]['step_name']
            ax.plot(sub.index, sub.values, 'o-', color=step_colors[step_num - 1],
                    linewidth=1.5, markersize=5, label=f'S{step_num}', alpha=0.8)
    ax.set_xlabel('Session Number')
    ax.set_ylabel('Composite Score')
    ax.set_title('Per-Step Learning Curves')
    ax.legend(fontsize=8, ncol=4, loc='lower right')
    ax.set_ylim(0, 5.5)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, 'learning_curve.png'), dpi=150, bbox_inches='tight')
    plt.show()
elif not df.empty:
    print(f"Only {df['session_id'].nunique()} session(s) so far. Complete at least 2 for progression analysis.")
else:
    print("No session data.")


## 8. Scenario Characteristics vs. Scores

**Question:** Are some scenario archetypes or situations harder than others?

This controls for the possibility that score improvements are just due to easier scenarios.


In [ ]:
if not df.empty and len(df) >= 5:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # By archetype
    ax = axes[0]
    arch_scores = df.groupby('archetype')['composite_score'].agg(['mean', 'count'])
    arch_scores = arch_scores[arch_scores['count'] >= 1].sort_values('mean', ascending=True)
    colors = ['#dc2626' if s < 3.0 else '#f59e0b' if s < 4.0 else '#16a34a' for s in arch_scores['mean']]
    ax.barh(arch_scores.index, arch_scores['mean'], color=colors)
    for i, (idx, row) in enumerate(arch_scores.iterrows()):
        ax.text(row['mean'] + 0.05, i, f'{row["mean"]:.1f} (n={int(row["count"])})', va='center', fontsize=9)
    ax.set_xlabel('Mean Composite Score')
    ax.set_title('Scores by Company Archetype')
    ax.set_xlim(0, 5.5)

    # By scope (product vs feature)
    ax = axes[1]
    scope_scores = df.groupby('scope')['composite_score'].agg(['mean', 'std', 'count'])
    ax.bar(scope_scores.index, scope_scores['mean'], yerr=scope_scores['std'],
           color=['#2563eb', '#7c3aed'], capsize=5, alpha=0.8)
    for i, (idx, row) in enumerate(scope_scores.iterrows()):
        ax.text(i, row['mean'] + row['std'] + 0.1, f'n={int(row["count"])}', ha='center', fontsize=10)
    ax.set_ylabel('Mean Composite Score')
    ax.set_title('Scores by Scenario Scope')
    ax.set_ylim(0, 5.5)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, 'scenario_characteristics.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Need more data for scenario characteristic analysis.")


## 9. Data Story Summary Statistics

Key numbers for the blog post narrative.


In [ ]:
if not df.empty and not df_long.empty:
    print("=" * 60)
    print("DATA STORY: KEY FINDINGS")
    print("=" * 60)

    n_sessions = df['session_id'].nunique()
    n_steps_total = len(df)
    n_models = len(MODEL_COLS)
    n_scores = len(df_long)
    models_used = list(MODEL_COLS.keys())

    print(f"\n📊 Dataset Size")
    print(f"   Practice sessions completed: {n_sessions}")
    print(f"   Total step evaluations: {n_steps_total}")
    print(f"   LLM models used: {n_models} ({', '.join(models_used)})")
    print(f"   Total model scores generated: {n_scores}")

    print(f"\n🎯 Scoring Overview")
    print(f"   Overall mean score: {df['composite_score'].mean():.2f}/5")
    print(f"   Score std dev: {df['composite_score'].std():.2f}")
    best_step = df.groupby('step_name')['composite_score'].mean().idxmax()
    worst_step = df.groupby('step_name')['composite_score'].mean().idxmin()
    print(f"   Strongest step: {best_step} ({df.groupby('step_name')['composite_score'].mean().max():.2f})")
    print(f"   Weakest step: {worst_step} ({df.groupby('step_name')['composite_score'].mean().min():.2f})")

    if n_models >= 2:
        print(f"\n🤝 Inter-Model Agreement")
        print(f"   Mean score spread (std dev): {df['agreement_std_dev'].mean():.2f}")
        print(f"   Mean score range: {df['agreement_range'].mean():.2f}")
        # Re-calculate alpha
        alpha_val = krippendorff_alpha(data_matrix) if 'data_matrix' in dir() else None
        if alpha_val is not None:
            print(f"   Krippendorff's alpha: {alpha_val:.3f}")

    if df_long['latency'].notna().any():
        print(f"\n⏱  Latency")
        for model in models_used:
            sub = df_long[(df_long['model'] == model) & df_long['latency'].notna()]
            if len(sub) > 0:
                print(f"   {model}: {sub['latency'].mean():.1f}s avg ({sub['latency'].min():.1f}-{sub['latency'].max():.1f}s)")

    if n_sessions >= 2:
        first_session = df[df['session_id'] == session_order.index[0]]['composite_score'].mean()
        last_session = df[df['session_id'] == session_order.index[-1]]['composite_score'].mean()
        delta = last_session - first_session
        print(f"\n📈 Learning Progression")
        print(f"   First session avg: {first_session:.2f}/5")
        print(f"   Latest session avg: {last_session:.2f}/5")
        print(f"   Improvement: {delta:+.2f} points")

    print("\n" + "=" * 60)

    # Save summary to file
    summary_path = os.path.join(OUTPUTS_DIR, 'data_story_summary.txt')
    with open(summary_path, 'w') as f:
        f.write(f"LLM Coaching Data Story — Summary Statistics\n")
        f.write(f"Generated: {datetime.now().isoformat()}\n")
        f.write(f"Sessions: {n_sessions} | Steps: {n_steps_total} | Models: {', '.join(models_used)}\n")
        f.write(f"Overall mean: {df['composite_score'].mean():.2f}/5\n")
    print(f"\nSummary saved to {summary_path}")
else:
    print("Run NB08 sessions first to generate data for this analysis.")


## 10. Export Figures & Data for Blog Post

All figures have been saved to `data/outputs/nb09/`. The analysis above provides the statistical foundation for the data story:

**Blog narrative structure:**
1. **Hook:** "I built an AI coaching tool — then turned the coaches into the subject of study."
2. **Setup:** Describe the interview prep tool and multi-model scoring architecture.
3. **Finding 1:** Inter-model agreement (or lack thereof) — what does it mean when Claude, GPT, and Gemini disagree on what makes a good product analytics response?
4. **Finding 2:** Systematic scoring biases — verbosity reward, model severity differences.
5. **Finding 3:** Learning curve — did the coaching actually help improve performance?
6. **Takeaway:** When LLMs can (and can't) replace human evaluation in coaching and assessment contexts.

---
*Data generated by NB08 (interview practice). Analysis by NB09.*
